# Imports and data:

In [1]:
import pandas as pd
import numpy as np
import regex as re

In [2]:
w1 = pd.read_excel("data/WINNERS 2000-2010 - 05.xlsx")
w2 = pd.read_excel("data/WINNERS 2011-2020 - 05.xlsx")
w3 = pd.read_excel("data/WINNERS 2021-2025 - 05.xlsx")
w4 = pd.read_excel("data/WINNERS 2026-2030 - 05.xlsx")

In [3]:
winners = pd.concat([w1, w2, w3, w4], ignore_index=True)

winners.shape

(43457, 10)

In [4]:
winners.dtypes

Name                   object
School                 object
Year                    int64
City                   object
Award                  object
Division               object
Category               object
School / Company       object
Division / Position    object
Dance Category         object
dtype: object

In [5]:
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [6]:
winners["Division"].value_counts()

Division
Senior Age Division             2837
Junior Age Division             2604
Pre-Competitive Age Division    1102
Top 12 - Senior Age Division       1
Name: count, dtype: int64

In [7]:
winners["Category"].value_counts()

Category
Classical Dance Category                   2909
Contemporary Dance Category                2874
Ensembles                                   451
Special Awards                              446
Large Ensembles                             133
Pas De Deux                                  73
Small Ensembles                              67
Classical & Contemporary Dance Category      24
Duets & Trios                                 3
Name: count, dtype: int64

In [8]:
for i in winners["Division / Position"].value_counts().index:
    print(i)

Junior Age Division
Senior Age Division
Pre-Competitive Age Division
Large Ensembles
Small Ensembles
Ensembles
Special Awards
Pas De Deux
Duet/Trio
Pas De Deux (The Nutcracker Suite Pas De Deux)
Pas De Deux (Satanella Pas De Deux)
Pas De Deux (Le Corsaire Act III)
Pas De Deux (Talisman)
Pas De Deux (Harlequinade Pas De Deux)
Pas De Deux (Grand Pas Classique Pas De Deux)
Pas De Deux (C'Est La Vie)
Small Ensembles (Coppelia, Pas De Deux)
Pas De Deux (Le Corsaire Pas De Deux)
Pas De Deux (Diana & Acteon , Pas De Deux)
Pas De Deux (Le Corsaire)
Pas De Deux (Grand Pas Classique)
Pas De Deux (La Fille Mal Gardee)
Pas De Deux (Giselle)
Pas De Deux (Cavalry Halt)
Pas De Deux (Swan Lake Act III)
Pas De Deux (Swan Lake Black Swan Pas De Deux)
Pas De Deux (Don Quixote, Pas De Deux)
Pas De Deux (The Sleeping Beauty Grand Pas De Deux)
Pas De Deux (Flower Festival In Genzano)
Pas De Deux (The Nutcracker Suite)
Pas De Deux (The Flames of Paris)
Pas De Deux (The Sleeping Beauty Bluebird)
Pas De Deux (

In [9]:
winners[winners.Name.str.contains(",", na=False)][["Name", "Category", "Division / Position"]]

,Name,Category,Division / Position
6,"Kory Glatman, NJ",NaN,NaN
87,"Melissa Lorry, MA",Contemporary Dance Category,NaN
124,"Arizona Ballet School, AZ",Special Awards,NaN
186,"April Rae, MD",NaN,NaN
207,"Bills, Bills, Bills",Large Ensembles,NaN
...,...,...,...
41363,"Strong, Calm, Slow",NaN,Duet/Trio
41910,"Diana & Acteon , Pas De Deux",NaN,Small Ensembles
41912,"Diana & Acteon , Pas De Deux",NaN,Small Ensembles
43165,"Spin, Measure, Cut",NaN,Small Ensembles


In [10]:
winners["Dance Category"].value_counts()

Dance Category
Classical Dance Category                   17361
Contemporary Dance Category                11612
Ensembles                                   1793
Special Awards                               889
Pas De Deux                                  477
Large Ensembles                              198
Small Ensembles                              110
Classical & Contemporary Dance Category      104
Ensembles And Pas De Deux                     44
Duets & Pas De Deux                           28
Contemporary/Open Dance Category              17
Top 12                                        15
Traditional Dance Category                     9
Duet/Trio                                      9
Top 6                                          6
Ensembles & Pas De Deux                        3
Trio                                           1
Duet                                           1
Name: count, dtype: int64

In [11]:
winners.isna().sum()

Name                     554
School                 36132
Year                       0
City                       0
Award                    174
Division               36913
Category               36477
School / Company        8286
Division / Position    11327
Dance Category         10780
dtype: int64

- Some awards are for the school overall rather than a person
- Some awards are for the name of the dance rather than people
- School awards wouldn't have a Dance Category or Position, which would explain some entries being null
- NaNs in certain columns can actually tell you what kind of award is being given:
    - NaN in School/Company ; School Award
- Some people don't belong in a School/Company

# Data Cleanup:

- Whenever Name and School have NaNs, there are no rows where School/Company is empty
    - So: replace School with whatever is in the School/Company for that row

In [12]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
7717,NaN,NaN,2011,"Torrington, CT",Outstanding School Award,NaN,NaN,"Washington School of Ballet, DC, USA",NaN,Special Awards
7806,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,The Art of Classical Ballet,NaN,Special Awards
7807,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,America's Ballet School,NaN,Special Awards
7905,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Los Angeles Ballet Academy, CA",NaN,Special Awards
7906,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Westlake School for the Performing Arts, CA",NaN,Special Awards
...,...,...,...,...,...,...,...,...,...,...
42644,NaN,NaN,2026,"Boca Raton, FL",Outstanding School Award,NaN,NaN,"Stars Dance Studio, FL",Special Awards,NaN
42928,NaN,NaN,2026,"Indianapolis, IN",Outstanding School Award,NaN,NaN,"Indiana Ballet Conservatory, IN",Special Awards,NaN
43200,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"N&D Ballet, MA",Special Awards,NaN
43201,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"Koltun Ballet Boston, MA",Special Awards,NaN


In [13]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna()) &
    (winners["School / Company"].isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category


In [14]:
mask = (winners["Name"].isna()) & (winners["School"].isna())

winners.loc[mask, "School"] = winners.loc[mask, "School / Company"]
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [15]:
mask =  (winners.School.isna()) & (~winners["School / Company"].isna())
winners.loc[
    mask,
    "School"    
] = winners.loc[mask, "School / Company"]

- Removing hanging whitespace from all string columns since it will effect analysis
- Turn everything uppercase, since there are inconsistent capitalization (or lack thereof)

In [16]:
winners["Name"] = winners.Name.str.strip()

In [17]:
str_cols = winners.select_dtypes(include=["object", "string"]).columns

for col in str_cols:
    mask = winners[col].notna()
    winners.loc[mask, col] = (
        winners.loc[mask, col]
        .astype(str)
        .str.replace(r"\s+,", ",", regex=True)
        .str.replace(r",\s+,", ",", regex=True)
        .str.replace(r"\(\s+(\d+)", r"(\1", regex=True)
        .str.replace(r"(\d+)\s+\)", r"\1)", regex=True)
    )

In [18]:
winners = winners.apply(lambda col: col.str.upper() if col.dtype == "object" else col)

In [19]:
names = sorted(winners.Name.dropna().unique())

with open("data/yagp_names.txt", "w") as file:
    file.writelines(
        name + "\n" for name in names
    )

- Inconsistent patterns for recording names:
    - First Name Last Name(s)
    - Last Name, First Name
    - First Name Last Name, [US state] (sometimes followed by what US state they're from) 
    - First Name Middle Name(s) and Last Name(s)
    - [1st person] & [2nd person]
    - [1st person], [2nd person]
    - [1st person] (age), [2nd person] (age)
    - (age)
    - [1st person] age, [2nd person] age
    - First Name Last Name (age), (dance that the person/people performed)
    - dance that the person/people performed, ([1st person], [2nd person])

In [20]:
winners[winners.Name.str.contains("& &", na=False)]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
31334,"ULIANA EFIMOVA & & MAKSIM APATONOV, (THE SLEEP...","ARMIDA BALLET SCHOOL, RUSSIA",2020,"PARIS, FRANCE",1ST PLACE,NaN,NaN,"ARMIDA BALLET SCHOOL, RUSSIA",NaN,PAS DE DEUX
31335,"ANA NOVAIS & & TOMAS RUAO, (LA BAYADERE)","CONSERVATÓRIO DE DANÇA DO VALE DO SOUSA, PORTUGAL",2020,"PARIS, FRANCE",2ND PLACE,NaN,NaN,"CONSERVATÓRIO DE DANÇA DO VALE DO SOUSA, PORTUGAL",NaN,PAS DE DEUX
31336,"ANA FRANCISCA CRUZ & & DIOGO BETENCOURT, (SATA...","DNA- DANCE N´ARTS SCHOOL, PORTUGAL",2020,"PARIS, FRANCE",3RD PLACE,NaN,NaN,"DNA- DANCE N´ARTS SCHOOL, PORTUGAL",NaN,PAS DE DEUX


In [21]:
def split_names(string):
    if pd.isna(string):
        return string  # keep NaN
    
    if isinstance(string, str):
        return [s.strip() for s in string.split("&")]
    
    return string

winners.Name = winners.Name.str.replace(r"& &", "&")
winners.Name = winners.Name.apply(split_names)
winners = winners.explode("Name", ignore_index=True)

In [22]:
names = winners[winners.Name.str.contains(",", na=False) == False].Name.unique()

winners["Names_resolved"] = winners["Name"].str.split(",")

winners[~winners.Names_resolved.isna()]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category,Names_resolved
2,8+3,INWOOD DANCE COMPANY,2000,"WASHINGTON, DC",3RD PLACE (TIE),NaN,ENSEMBLES,NaN,NaN,NaN,[8+3]
3,CABARET,"DANCE CONSERVATORY, DE",2000,"WASHINGTON, DC",3RD PLACE (TIE),NaN,ENSEMBLES,NaN,NaN,NaN,[CABARET]
4,MAKE THEM HEAR YOU,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",2ND PLACE,NaN,ENSEMBLES,NaN,NaN,NaN,[MAKE THEM HEAR YOU]
5,CAPRICE,"DANCE CONSERVATORY, DE",2000,"WASHINGTON, DC",1ST PLACE,NaN,ENSEMBLES,NaN,NaN,NaN,[CAPRICE]
6,"KORY GLATMAN, NJ",NaN,2000,"WASHINGTON, DC",3RD PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,"[KORY GLATMAN, NJ]"
...,...,...,...,...,...,...,...,...,...,...,...
43515,GERDA'S RIVER JOURNEY,"BAYER BALLET ACADEMY, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"BAYER BALLET ACADEMY, CA",LARGE ENSEMBLES,NaN,[GERDA'S RIVER JOURNEY]
43516,DANCE FOR SIX,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",LARGE ENSEMBLES,NaN,[DANCE FOR SIX]
43517,COPPELIA FRIENDS DANCE,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",LARGE ENSEMBLES,NaN,[COPPELIA FRIENDS DANCE]
43518,KIRA FARGAS,FOR ALL THE WORK,2026,"SAN FRANCISCO, CA (MAR)",OUTSTANDING CHOREOGRAPHER AWARD,NaN,NaN,FOR ALL THE WORK,SPECIAL AWARDS,NaN,[KIRA FARGAS]


In [23]:
winners.loc[
    ((winners.Names_resolved.str[1].str.strip() + " " + winners.Names_resolved.str[0].str.strip()).isin(names)), 
    ["Names_resolved"]
] = winners.Names_resolved.str[1].str.strip() + " " + winners.Names_resolved.str[0].str.strip()

winners.loc[
    ((winners.Names_resolved.str[0].str.strip() + " " + winners.Names_resolved.str[1].str.strip()).isin(names)), 
    ["Names_resolved"]
] = winners.Names_resolved.str[0].str.strip() + " " + winners.Names_resolved.str[1].str.strip()

In [24]:
for i in winners.Names_resolved[~winners.Names_resolved.isna()].value_counts().index:
    if not isinstance(i, list):
        print(i)

COPPELIA PAS DE DEUX
DON QUIXOTE PAS DE DEUX
ALEXANDRIA MARX
HARLEQUINADE PAS DE DEUX
RACHEL ROHRICH
SWAN LAKE PAS DE TROIS
ODALISQUES LE CORSAIRE
RYLAND ACREE III
LAUREN LEB
GENEVIEVE COOVREY
JASMINE JIMISON
SILVI LYBBERT
JESLYN CHEN
PAS DE DEUX GRAND PAS CLASSIQUE
MASUMI YOSHIMOTO
ORLANDO BALLET SCHOOL FL
ALL AMERICAN CLASSICAL BALLET SCHOOL FL


In [25]:
mask = winners["Names_resolved"].apply(lambda x: isinstance(x, str))

winners.loc[mask, "Name"] = winners.loc[mask, "Names_resolved"]
winners.drop(columns="Names_resolved", inplace=True)
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",OUTSTANDING SCHOOL AWARD,NaN,SPECIAL AWARDS,NaN,NaN,NaN
1,NaN,"THE KIROV ACADEMY, DC",2000,"WASHINGTON, DC",OUTSTANDING SCHOOL AWARD,NaN,SPECIAL AWARDS,NaN,NaN,NaN
2,8+3,INWOOD DANCE COMPANY,2000,"WASHINGTON, DC",3RD PLACE (TIE),NaN,ENSEMBLES,NaN,NaN,NaN
3,CABARET,"DANCE CONSERVATORY, DE",2000,"WASHINGTON, DC",3RD PLACE (TIE),NaN,ENSEMBLES,NaN,NaN,NaN
4,MAKE THEM HEAR YOU,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",2ND PLACE,NaN,ENSEMBLES,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43516,DANCE FOR SIX,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",LARGE ENSEMBLES,NaN
43517,COPPELIA FRIENDS DANCE,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",LARGE ENSEMBLES,NaN
43518,KIRA FARGAS,FOR ALL THE WORK,2026,"SAN FRANCISCO, CA (MAR)",OUTSTANDING CHOREOGRAPHER AWARD,NaN,NaN,FOR ALL THE WORK,SPECIAL AWARDS,NaN
43519,HADASSAH PERRY,FOR ALL THE WORK,2026,"SAN FRANCISCO, CA (MAR)",OUTSTANDING CHOREOGRAPHER AWARD,NaN,NaN,FOR ALL THE WORK,SPECIAL AWARDS,NaN


- Misspellings/errors that need to be counted as a person:
    - ALANAGRIFFITH = ALANA GRIFFITH
    - ALTHEA JOHNSON - STAUB
    - ANDREAJONUSAS = ANDREA JONUSAS
    - ELISA D[CARPIO = ELISA D'CARPIO
    - GABRIELLEGIRARD = GABRIELLE GIRARD
    - LOGANHILLMAN = LOGAN HILLMAN
    - MAKAI LEWIS - REES
    - MARIARODRIGUEZ = MARIA RODRIGUEZ
    - PHILIP MARTIN‐NIELSON
    - VICTORIAHASK = VICTORIA HASK
    - WING YAN [IBBY] CHOW

In [26]:
winners.loc[
    (winners.Name.str.contains("ALANAGRIFFITH", na=False)),
    "Name"
] = "ALANA GRIFFITH"

winners.loc[
    (winners.Name.str.contains("ANDREAJONUSAS", na=False)),
    "Name"
] = "ANDREA JONUSAS"

winners.loc[
    (winners.Name.str.contains("ELISA D[CARPIO", regex=False, na=False)),
    "Name"
] = "ELISA D'CARPIO"

winners.loc[
    (winners.Name.str.contains("GABRIELLEGIRARD", na=False)),
    "Name"
] = "GABRIELLE GIRARD"

winners.loc[
    (winners.Name.str.contains("LOGANHILLMAN", na=False)),
    "Name"
] = "LOGAN HILLMAN"

winners.loc[
    (winners.Name.str.contains("MARIARODRIGUEZ", na=False)),
    "Name"
] = "MARIA RODRIGUEZ"

winners.loc[
    (winners.Name.str.contains("VICTORIAHASK", na=False)),
    "Name"
] = "VICTORIA HASK"

winners.loc[
    (winners.Name.str.contains("ALANA GRIFFITH", na=False)) |
    (winners.Name.str.contains("ALTHEA JOHNSON - STAUB", na=False)) |
    (winners.Name.str.contains("ANDREA JONUSAS", na=False)) |
    (winners.Name.str.contains("ELISA D'CARPIO", na=False)) |
    (winners.Name.str.contains("GABRIELLE GIRARD", na=False)) |
    (winners.Name.str.contains("LOGAN HILLMAN", na=False)) |
    (winners.Name.str.contains("MARIA RODRIGUEZ", na=False)) |
    (winners.Name.str.contains("MAKAI LEWIS - REES", na=False)) |
    (winners.Name.str.contains("PHILIP MARTIN‐NIELSON", na=False)) |
    (winners.Name.str.contains("VICTORIA HASK", na=False)) |
    (winners.Name.str.contains("WING YAN [IBBY] CHOW", na=False)), 
    "is_name"
] = "person"

- Classifying what entries have people's names:

In [27]:
# winners.is_name = winners["Name"].apply(...)
# winners.is_name.value_counts()

- Cleaning up the names column to just get people's names:

In [28]:
winners[winners.Name.str.contains(",", na=False)].Name.unique()

array(['KORY GLATMAN, NJ', 'MELISSA LORRY, MA',
       'ARIZONA BALLET SCHOOL, AZ', 'APRIL RAE, MD',
       'BILLS, BILLS, BILLS', 'RITA STROM, CA', 'SACHA DE SOLA, FL',
       'MEGAN CARLO, FL', 'BROTHER, CAN YOU SPARE A DIME',
       'ADAMS SCHOOL OF DANCE, VT', 'PAS DE DEUX, "GISELLE", ACT III',
       'GRANDE PAS DE DEUX, LE CORSAIRE', 'ENGLISH STRING SUITE, 4TH',
       'ODALISQUE, LE CORSAIRE', 'CONCERTO GROSSO, 4TH MOVEMENT',
       'LA FILLE MAL GARDEE, THE CLOGS DANCE', 'MIRTH, FIRST MOVEMENT',
       'WHEAT PAS DE DEUX, COPPELIA', 'SODA BALLET ACADEMY, JAPAN',
       'ROCHESTER, NY', 'PERHAPS, PERHAPS, PERHAPS',
       'KENNETH DUANE SHELBY, JR', 'COPPELIA, THE DOLL',
       'KENNETH SHELBY, JR', 'KINSEY NOVAK, 13', 'CAROLIE, LAUVETZ',
       'IMPACT DANCE OF ATLANTA, GA',
       'MORNING STAR DANCE ACADEMY OF ATLANTA, GA',
       'INTERNATIONAL BALLET ACADEMY, NC', 'AKHMEDOVA BALLET ACADEMY, MD',
       'INTERNATIONAL CITY SCHOOL OF BALLET, GA',
       'MARIA CLARA COELHO (1

In [29]:
winners.Name = winners.Name.str.split(",")
winners = winners.explode("Name")

In [30]:
winners.Name = winners.Name.str.strip()
winners.Name.value_counts()

Name
FACULTY                         43
CAITLYN DIONEDA                 36
KATIA ALMAYEVA                  33
LE CORSAIRE                     32
PAS DE DEUX FROM LE CORSAIRE    31
                                ..
ELLA HU                          1
ISABELLE ALVAREZ                 1
TAYLOR OTSUKI                    1
KATHARINE ZHONG                  1
LUKA MCKINNEY                    1
Name: count, Length: 16761, dtype: int64

In [31]:
# for name in sorted(winners.Name.dropna().unique()):
#     print(name)

In [32]:
# for name in winners.Name.dropna().unique():
#     print(name)

In [33]:
for name in sorted(
    winners[
        (winners.Name.str.contains(",", na=False)) & 
        (winners.is_name == "person") 
    ]["Name"].unique()
):
    print(name)

In [34]:
# for name in sorted(winners[
#     (winners.is_name == "person") |
#     (winners.is_name == "people")
# ]["Name"].unique()):
#     print(name)

- Dancers:
    - classifications: person, people

In [35]:
dancers = winners[
    (winners.is_name == "person")
].copy()

mask = (dancers["is_name"] == "people") | (dancers.is_name == "person")
dancers.loc[mask, "Name"] = dancers.loc[mask, "Name"].str.split(",")
dancers = dancers.explode("Name")
dancers["Name"] = dancers["Name"].str.strip()

dancers

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category,is_name
103,ANDREA JONUSAS,"LIANA'S SELECTED WORKSHOP, FL",2000,"BOCA RATON, FL",1ST PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,person
369,ANDREA JONUSAS,LIANA NAVARRO SELECTED WORKSHOP,2002,"WINSTON-SALEM, NC",TOP 12,JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,person
375,ANDREA JONUSAS,NaN,2002,"WINSTON-SALEM, NC",2ND PLACE (TIE),JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,person
999,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2003,"GREENVILLE, SC",TOP 12,JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,person
1004,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2003,"GREENVILLE, SC",2ND PLACE,JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,person
1015,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2003,"GREENVILLE, SC",TOP 12,JUNIOR AGE DIVISION,CLASSICAL DANCE CATEGORY,NaN,NaN,NaN,person
1019,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2003,"GREENVILLE, SC",2ND PLACE,JUNIOR AGE DIVISION,CLASSICAL DANCE CATEGORY,NaN,NaN,NaN,person
1668,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2004,"GREENVILLE, SC",TOP 12,JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,person
1672,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2004,"GREENVILLE, SC",1ST PLACE,JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,person
1678,ANDREA JONUSAS,"ARTISTIC DANCE CENTER, FL, USA",2004,"GREENVILLE, SC",TOP 12,JUNIOR AGE DIVISION,CLASSICAL DANCE CATEGORY,NaN,NaN,NaN,person


In [36]:
# for name in sorted(dancers.Name.unique()):
#     print(name)